In [1]:
!git clone https://github.com/keiranoluv/final_project
!git clone https://github.com/PaddlePaddle/PaddleOCR.git

Cloning into 'final_project'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 33 (delta 6), reused 28 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 16.21 KiB | 5.40 MiB/s, done.
Resolving deltas: 100% (6/6), done.
Cloning into 'PaddleOCR'...
remote: Enumerating objects: 353017, done.
remote: Counting objects: 100% (1141/1141), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 353017 (delta 1064), reused 991 (delta 991), pack-reused 351876 (from 3)
Receiving objects: 100% (353017/353017), 1.87 GiB | 33.61 MiB/s, done.
Resolving deltas: 100% (279208/279208), done.


In [2]:
%cd /kaggle/working/PaddleOCR

!python -m pip install -q -r requirements.txt
!python -m pip install -q paddlepaddle-gpu==3.3.0 \
  -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ \
  --no-deps

/kaggle/working/PaddleOCR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 97.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 809.9 kB/s eta 0:00:00


In [3]:
import csv
from pathlib import Path

DATASET_ROOT = Path("/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2")
OUTPUT_ROOT = Path("/kaggle/working/mthv2_labels")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for split in ["train", "val", "test"]:
    src = DATASET_ROOT / f"{split}.tsv"
    dst = OUTPUT_ROOT / f"{split}.txt"

    count = 0

    with src.open("r", encoding="utf-8") as fin, \
         dst.open("w", encoding="utf-8") as fout:

        reader = csv.DictReader(fin, delimiter="\t")

        for row in reader:
            fout.write(f'{row["image_path"]}\t{row["text"]}\n')
            count += 1

    print(f"{split}: {count:,} samples -> {dst}")

train: 72,563 samples -> /kaggle/working/mthv2_labels/train.txt
val: 7,753 samples -> /kaggle/working/mthv2_labels/val.txt
test: 25,262 samples -> /kaggle/working/mthv2_labels/test.txt


In [4]:
TRAIN_CHARS = Path(
    "/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2/train_characters.txt"
)

PADDLE_DICT = Path(
    "/kaggle/working/PaddleOCR/ppocr/utils/dict/ppocrv5_dict.txt"
)

train_chars = {
    line.rstrip("\r\n")
    for line in TRAIN_CHARS.open("r", encoding="utf-8")
}

paddle_chars = {
    line.rstrip("\r\n")
    for line in PADDLE_DICT.open("r", encoding="utf-8")
}

# bỏ dòng rỗng nhưng GIỮ space nếu dataset thực sự có
train_chars.discard("")
paddle_chars.discard("")

missing = train_chars - paddle_chars
covered = train_chars & paddle_chars

print("=== Character coverage ===")
print(f"MTHv2 train unique chars : {len(train_chars):,}")
print(f"PP-OCRv5 dict chars      : {len(paddle_chars):,}")
print(f"Covered                  : {len(covered):,}")
print(f"Missing                  : {len(missing):,}")
print(f"Coverage                 : {len(covered) / len(train_chars) * 100:.4f}%")

=== Character coverage ===
MTHv2 train unique chars : 6,063
PP-OCRv5 dict chars      : 18,383
Covered                  : 5,277
Missing                  : 786
Coverage                 : 87.0361%


## B1 – Fine-tuning với dictionary mặc định của PP-OCRv5

B1 fine-tune mô hình `PP-OCRv5_server_rec` trên tập MTHv2 nhưng vẫn giữ nguyên **character dictionary mặc định của PP-OCRv5**.

### Character coverage

Tập train MTHv2 có:

- **6,063** ký tự khác nhau.
- Dictionary mặc định của PP-OCRv5 bao phủ **5,277** ký tự.
- Tương đương khoảng **87.04% unique-character coverage**.
- Có **786 ký tự** trong tập train không nằm trong vocabulary mặc định.

Các ký tự ngoài vocabulary này được gọi là **OOV (Out Of Vocabulary)**.

Xét theo số lần xuất hiện:

- Tổng số ký tự trong tập train: **725,254**
- Số lần xuất hiện của các ký tự OOV: **10,090**
- Tỷ lệ OOV theo số lần xuất hiện: khoảng **1.39%**

### Ảnh hưởng của OOV trong PaddleOCR

PaddleOCR sử dụng `character_dict_path` để ánh xạ từng ký tự trong ground truth sang chỉ số của vocabulary.

Nếu một ký tự không tồn tại trong dictionary, label encoder sẽ bỏ qua ký tự đó thay vì dừng chương trình.

Ví dụ:

```text
Ground truth gốc:
天地䖏玄黃

Nếu ký tự 䖏 không nằm trong dictionary:
天地玄黃

In [5]:
!mkdir -p pretrained

!wget -O pretrained/PP-OCRv5_server_rec_pretrained.pdparams \
  https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_rec_pretrained.pdparams

--2026-08-07 20:54:10--  https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_rec_pretrained.pdparams
Resolving paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:913:0:ff:b0a4:a156
Connecting to paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 214594738 (205M) [application/octet-stream]
Saving to: ‘pretrained/PP-OCRv5_server_rec_pretrained.pdparams’

pretrained/PP-OCRv5 100%[===================>] 204.65M  16.6MB/s    in 14s     

2026-08-07 20:54:26 (14.3 MB/s) - ‘pretrained/PP-OCRv5_server_rec_pretrained.pdparams’ saved [214594738/214594738]



In [6]:
%cd /kaggle/working/PaddleOCR

!python -m paddle.distributed.launch \
  --gpus "0,1" \
  tools/train.py \
  -c configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml \
  -o \
  Global.pretrained_model=./pretrained/PP-OCRv5_server_rec_pretrained.pdparams \
  Global.epoch_num=20 \
  Global.save_model_dir=/kaggle/working/final_project/outputs/B1_20epochs_2gpu_bs64 \
  Global.eval_batch_step="[0,500]" \
  Train.loader.batch_size_per_card=64 \
  Train.sampler.first_bs=64 \
  Train.dataset.data_dir=/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2 \
  Train.dataset.label_file_list='["/kaggle/working/mthv2_labels/train.txt"]' \
  Eval.dataset.data_dir=/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2 \
  Eval.dataset.label_file_list='["/kaggle/working/mthv2_labels/val.txt"]'

/kaggle/working/PaddleOCR
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
LAUNCH INFO 2026-08-07 20:54:30,227 -----------  Configuration  ----------------------
LAUNCH INFO 2026-08-07 20:54:30,227 auto_cluster_config: 0
LAUNCH INFO 2026-08-07 20:54:30,227 auto_parallel_config: None
LAUNCH INFO 2026-08-07 20:54:30,227 auto_tuner_json: None
LAUNCH INFO 2026-08-07 20:54:30,227 devices: 0,1
LAUNCH INFO 2026-08-07 20:54:30,227 elastic_level: -1
LAUNCH INFO 2026-08-07 20:54:30,227 elastic_timeout: 30
LAUNCH INFO 2026-08-07 20:54:30,228 enable_gpu_log: True
LAUNCH INFO 2026-08-07 20:54:30,228 gloo_port: 6767
LAUNCH INFO 2026-08-07 20:54:30,228 host: None
LAUNCH INFO 2026-08-07 20:54:30,228 ips: None
LAUNCH INFO 2026-08-07 